# Workflow

1. Texts converted into sentence embeddings
2. Dimensionality reduction with UMAP
3. Clustering with HDBSCAN
4. c-TF-IDF
5. Representation model

## Input texts

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired
import nbformat

texts = ["The consultant team were really responsive to my questions.  Several different doctors explained things when I couldn't speak to my consultant. Please send my compliments to the team.",
        "The nursing staff were too busy.  When I pressed the call bell, it took a long time for someone to come.  I had to wait for ages before I got any help with my pain, and going to the toilet.  When I didn't get my pain medication on time, I couldn't even get out of bed to go to the toilet, and I had an accident.  I was really embarrassed and upset about that.  I wish the staff had been able to help me sooner.",
        "The food was really bad.  Mum doesn't eat much at the best of times, but the food was so bad that she hardly ate anything at all.  How is she going to get better if she doesn't have the basics like food and water?  I don't understand how the hospital can let patients go hungry like that.  It's really unacceptable.",
        "I was skeptical about using the app to book my appointment, but it was actually really easy to use.  I was able to find a time that worked for me and book my appointment in just a few clicks.  I wasn't waiting on letters that arrive too late, and I knew I wasn't lost in the system.  It's a big improvment.",
        "The time I waited for an appointment was a joke.  Why does it take so long to get an appointment?  I had to wait for months before I could see a doctor, and when I finally got an appointment, it was cancelled at the last minute.  It's really frustrating and unacceptable.  I don't understand why the healthcare system can't do better than this.",
        "The staff were really kind and helpful.  They went out of their way to make sure I was comfortable and had everything I needed.  I felt like they really cared about me and my wellbeing, and that made a big difference in my experience at the hospital.  I'm really grateful for their kindness and support.",
        "Dad was really impressed with the care he received.  The doctors and nurses were really knowledgeable and professional, and they made sure to explain everything to him in a way that he could understand.  He felt like he was in good hands, and that gave him a lot of confidence in his treatment.  He's really grateful for the care he received.",
        "Dad doesn't speak English.  It's ok when I can be with him in the hospital, but I can't be there all the time as I've got kids and a job.  The hospital said they could provider and interpreter, but we didn't get one for the first few days, and when we did get one, they weren't very good.  They didn't understand Dad's accent, and they kept getting things wrong.  It was really frustrating and made it hard for Dad to communicate with the staff.  I wish the hospital had been able to provide better support for non-English speakers.",
        "I didn't understand what the doctors were saying.  They used a lot of medical jargon that I didn't understand, and they didn't take the time to explain things in a way that I could understand.  I felt really confused and overwhelmed, and it made it hard for me to make informed decisions about my care.  I wish the doctors had been able to communicate more clearly with me.",
        "I'm really angry that I was sent home so early.  I didn't feel ready to go home, and I felt like the hospital was just trying to get rid of me.  I had a lot of questions and concerns about my care, but the staff didn't take the time to address them.  I felt really abandoned and unsupported, and it made my recovery much harder than it needed to be.  I wish the hospital had been able to provide better support for patients who are being discharged."
        "I was sat in a corridor for nearly 2 days.  I was in pain, on my own, and it was very undignified.  I felt sorry for the staff, who were clearly overwhelmed, but I wish they had been able to find a better solution for me.  Why doesn't the hospital open more beds?"
]


## Sentence embedding model

In [2]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(texts, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.79it/s]


## Dimensionality reduction with UMAP

In [3]:
umap_model = UMAP(
    n_neighbors=3,
    n_components=2,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

## Clustering: HDBSCAN

In [4]:
hdbscan_model = HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

## c-TF-IDF

In [5]:
vectorizer_model = CountVectorizer(stop_words="english",
                                   ngram_range=(1, 2))

ctfidf_model = ClassTfidfTransformer()

## Representation Model

In [6]:
representation_model = KeyBERTInspired()

## Build BERTopic model

In [7]:
topic_model = BERTopic(
                      embedding_model=embedding_model,
                      umap_model=umap_model,
                      hdbscan_model=hdbscan_model,
                      vectorizer_model=vectorizer_model,
                      ctfidf_model=ctfidf_model,
                      representation_model=representation_model,
                      calculate_probabilities=True,
                      verbose=True
                      )

## Fit model

In [8]:
topics, probabilities = topic_model.fit_transform(texts)

2026-06-02 17:24:12,628 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 1/1 [00:00<00:00,  6.98it/s]
2026-06-02 17:24:12,781 - BERTopic - Embedding - Completed ✓
2026-06-02 17:24:12,782 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-02 17:24:20,368 - BERTopic - Dimensionality - Completed ✓
2026-06-02 17:24:20,369 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-02 17:24:20,372 - BERTopic - Cluster - Completed ✓
2026-06-02 17:24:20,374 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-02 17:24:20,568 - BERTopic - Representation - Completed ✓


## View results

In [9]:


# Visualise documents
fig = topic_model.visualize_documents(docs=texts, topics=topics, embeddings=embeddings)
fig.show()

In [10]:
topic_model.visualize_topics()

In [11]:
topic_model.visualize_barchart(top_n_topics=5)

In [12]:
topic_model.visualize_heatmap()

In [13]:
topic_info = topic_model.get_topic_info()
print(topic_info)

for topic_id in topic_info.Topic:
    print(f"Topic {topic_id}:")
    print(topic_model.get_topic(topic_id))



   Topic  Count                                               Name  \
0      0      5  0_waited appointment_appointment_patients_book...   
1      1      3  1_experience hospital_hospital_hospital really...   
2      2      2  2_understand doctors_doctors explained_wish do...   

                                      Representation  \
0  [waited appointment, appointment, patients, bo...   
1  [experience hospital, hospital, hospital reall...   
2  [understand doctors, doctors explained, wish d...   

                                 Representative_Docs  
0  [The food was really bad.  Mum doesn't eat muc...  
1  [The staff were really kind and helpful.  They...  
2  [The consultant team were really responsive to...  
Topic 0:
[('waited appointment', np.float32(0.5288178)), ('appointment', np.float32(0.5183519)), ('patients', np.float32(0.50097626)), ('book appointment', np.float32(0.4501449)), ('doctor finally', np.float32(0.41680488)), ('hospital', np.float32(0.39460218)), ('doctor', n

In [16]:
# Add results back to dataframe
import pandas as pd

df = pd.DataFrame({"text": texts, "topic": topics})
print(df)

                                                text  topic
0  The consultant team were really responsive to ...      2
1  The nursing staff were too busy.  When I press...      0
2  The food was really bad.  Mum doesn't eat much...      0
3  I was skeptical about using the app to book my...      0
4  The time I waited for an appointment was a jok...      0
5  The staff were really kind and helpful.  They ...      1
6  Dad was really impressed with the care he rece...      1
7  Dad doesn't speak English.  It's ok when I can...      1
8  I didn't understand what the doctors were sayi...      2
9  I'm really angry that I was sent home so early...      0


In [17]:
# Return keywords for each topic
topic_info = topic_model.get_topic_info()

for topic_id in topic_info.Topic:
    if topic_id == -1:
        continue

    keywords = topic_model.get_topic(topic_id)
    print(f"Topic {topic_id}: {keywords}")

Topic 0: [('waited appointment', np.float32(0.5288178)), ('appointment', np.float32(0.5183519)), ('patients', np.float32(0.50097626)), ('book appointment', np.float32(0.4501449)), ('doctor finally', np.float32(0.41680488)), ('hospital', np.float32(0.39460218)), ('doctor', np.float32(0.38044894)), ('doesn hospital', np.float32(0.35523736)), ('care staff', np.float32(0.35033858)), ('concerns care', np.float32(0.32403183))]
Topic 1: [('experience hospital', np.float32(0.43263385)), ('hospital', np.float32(0.41914913)), ('hospital really', np.float32(0.41790265)), ('dad communicate', np.float32(0.38956544)), ('hospital said', np.float32(0.37923425)), ('communicate staff', np.float32(0.3775648)), ('grateful care', np.float32(0.37501246)), ('doctors nurses', np.float32(0.36435544)), ('hospital time', np.float32(0.362476)), ('kindness support', np.float32(0.34988582))]
Topic 2: [('understand doctors', np.float32(0.47061074)), ('doctors explained', np.float32(0.43792003)), ('wish doctors', np.

In [18]:
# Return only top N keywords for each topic
for topic_id in topic_model.get_topic_info().Topic:
    if topic_id == -1:
        continue

    top_words = [
        word for word, score in topic_model.get_topic(topic_id)[:5]
    ]

    print(f"Topic {topic_id}: {top_words}")

Topic 0: ['waited appointment', 'appointment', 'patients', 'book appointment', 'doctor finally']
Topic 1: ['experience hospital', 'hospital', 'hospital really', 'dad communicate', 'hospital said']
Topic 2: ['understand doctors', 'doctors explained', 'wish doctors', 'confused overwhelmed', 'different doctors']


In [ ]:
# Return topic labels in a dataframe
topic_info = topic_model.get_topic_info()

topic_info["keywords"] = (
    topic_info["Topic"].apply(
        lambda x: ", ".join([
            word for word, _ in topic_model.get_topic(x)[:5]
        ])
        if x != -1 else "Outliers"
    )
)

print(topic_info[["Topic", "Count", "keywords"]])

   Topic  Count                                           keywords
0      0      5  waited appointment, appointment, patients, boo...
1      1      3  experience hospital, hospital, hospital really...
2      2      2  understand doctors, doctors explained, wish do...


In [19]:
# Generate more meaningful topic labels

topic_model.generate_topic_labels()

['0_waited appointment_appointment_patients',
 '1_experience hospital_hospital_hospital really',
 '2_understand doctors_doctors explained_wish doctors']

In [20]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,5,0_waited appointment_appointment_patients_book...,"[waited appointment, appointment, patients, bo...",[The food was really bad. Mum doesn't eat muc...
1,1,3,1_experience hospital_hospital_hospital really...,"[experience hospital, hospital, hospital reall...",[The staff were really kind and helpful. They...
2,2,2,2_understand doctors_doctors explained_wish do...,"[understand doctors, doctors explained, wish d...",[The consultant team were really responsive to...
